### Importing the Dataset

In [1]:
import seaborn as sns
df = sns.load_dataset('tips')

In [2]:
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [4]:
df['sex'].value_counts()

sex
Male      157
Female     87
Name: count, dtype: int64

In [5]:
df['smoker'].value_counts()

smoker
No     151
Yes     93
Name: count, dtype: int64

In [6]:
df['day'].value_counts()

day
Sat     87
Sun     76
Thur    62
Fri     19
Name: count, dtype: int64

In [7]:
df['time'].value_counts()

time
Dinner    176
Lunch      68
Name: count, dtype: int64

### Dependent and Indepenent Feature

In [8]:

df.columns
X = df[['tip', 'sex', 'smoker', 'day', 'time', 'size']]
y = df['total_bill']

### Train Test Split 

In [9]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.25,random_state = 10)

### Feature Encoding(Label Encoding and One Hot Encoding)
#### Label Encoding - sex , smoker , time  (for binary categories)

#### One Hot Encoding - day (for multi-category features without order)

### Implementing Label Encoding

In [10]:
from sklearn.preprocessing import LabelEncoder
le1 = LabelEncoder()
le2 = LabelEncoder()
le3 = LabelEncoder()

In [11]:
X_train['sex'] = le1.fit_transform(X_train['sex'])
X_train['smoker'] = le2.fit_transform(X_train['smoker'])
X_train['time'] = le3.fit_transform(X_train['time'])

In [12]:
X_train.head()

,tip,sex,smoker,day,time,size
58,1.76,1,1,Sat,0,2
1,1.66,1,0,Sun,0,3
2,3.50,1,0,Sun,0,3
68,2.01,1,0,Sat,0,2
184,3.00,1,1,Sun,0,2


In [13]:
X_test['sex'] = le1.fit_transform(X_test['sex'])
X_test['smoker'] = le2.fit_transform(X_test['smoker'])
X_test['time'] = le3.fit_transform(X_test['time'])

In [14]:
X_test.head()

,tip,sex,smoker,day,time,size
162,2.00,0,0,Sun,0,3
60,3.21,1,1,Sat,0,2
61,2.00,1,1,Sat,0,2
63,3.76,1,1,Sat,0,4
69,2.09,1,1,Sat,0,2


### Implementing OneHot Encoding

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [16]:
ct = ColumnTransformer(transformers =[('onehot',OneHotEncoder(drop='first'),[3])],remainder = 'passthrough' )

In [17]:
X_train = ct.fit_transform(X_train)

In [18]:
X_test = ct.fit_transform(X_test)

### SVR -- Support Vector Regressor

In [19]:
from sklearn.svm import SVR
svr = SVR()
svr.fit(X_train,y_train)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [20]:
y_pred = svr.predict(X_test)

In [21]:
from sklearn.metrics import r2_score,mean_absolute_error
print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))

0.46028114561159283
4.1486423210190235


### HyperParameter using GridSearchCV

In [22]:
from sklearn.model_selection import GridSearchCV

### Defining parameter range
param_grid = {'C':[0.1,1,10,100,1000],
            'gamma':[1,0.1,0.01,0.001,0.0001],
            'kernel':['rbg']}

In [23]:
grid = GridSearchCV(SVR(),param_grid = param_grid,refit= True,verbose = 3)

In [24]:
grid.fit(X_train,y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
[CV 1/5] END ..........C=0.1, gamma=1, kernel=rbg;, score=nan total time=   0.0s
[CV 2/5] END ..........C=0.1, gamma=1, kernel=rbg;, score=nan total time=   0.0s
[CV 3/5] END ..........C=0.1, gamma=1, kernel=rbg;, score=nan total time=   0.0s
[CV 4/5] END ..........C=0.1, gamma=1, kernel=rbg;, score=nan total time=   0.0s
[CV 5/5] END ..........C=0.1, gamma=1, kernel=rbg;, score=nan total time=   0.0s
[CV 1/5] END ........C=0.1, gamma=0.1, kernel=rbg;, score=nan total time=   0.0s
[CV 2/5] END ........C=0.1, gamma=0.1, kernel=rbg;, score=nan total time=   0.0s
[CV 3/5] END ........C=0.1, gamma=0.1, kernel=rbg;, score=nan total time=   0.0s
[CV 4/5] END ........C=0.1, gamma=0.1, kernel=rbg;, score=nan total time=   0.0s
[CV 5/5] END ........C=0.1, gamma=0.1, kernel=rbg;, score=nan total time=   0.0s
[CV 1/5] END .......C=0.1, gamma=0.01, kernel=rbg;, score=nan total time=   0.0s
[CV 2/5] END .......C=0.1, gamma=0.01, kernel=r

ValueError: 
All the 125 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
125 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\ANKIT\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\ANKIT\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py", line 1329, in wrapper
    estimator._validate_params()
  File "C:\Users\ANKIT\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py", line 492, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\ANKIT\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\_param_validation.py", line 98,

In [ ]:
y_pred1 = grid.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score,mean_absolute_error
print(r2_score(y_test,y_pred1))
print(mean_absolute_error(y_test,y_pred1))